# AHON FloodWatch: Water-Level Visualization and Regression Forecast

Run this notebook in Google Colab. First export the database from the repository root:

```powershell
.\backend\Django\venv\Scripts\python.exe scripts\export_water_level_readings.py --output artifacts\water_level_readings.csv
```

Upload `artifacts/water_level_readings.csv` when prompted, then run cells in order. The export contains only timestamps, water levels, and statuses; it does not include resident information. The generated CSV is git-ignored.

The notebook uses recorded readings only; it does not generate synthetic training data. The current sensor calibration is 0-600 ADC = 0-4 cm. It excludes and reports values outside 0-4 cm because older records may contain raw ADC values incorrectly stored as centimeters. Warning begins at 2 cm; Critical/Danger begins at 3 cm.

The regression demo predicts water level 5 minutes ahead by default. Configure the interval and horizon below. Results are exploratory and must not be used as the sole basis for evacuation decisions.

In [ ]:
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RESAMPLE_SECONDS = 30
FORECAST_MINUTES = 5
TEST_FRACTION = 0.20
MAX_SENSOR_CM = 4.0
WARNING_CM = 2.0
DANGER_CM = 3.0
MIN_SUPERVISED_ROWS = 80
GAP_MULTIPLIER = 1.5

print(f'Resample interval: {RESAMPLE_SECONDS} seconds')
print(f'Forecast horizon: {FORECAST_MINUTES} minutes')

## 1. Upload the exported readings

The CSV must contain `timestamp` and `water_level_cm`; `status` is optional. The export helper in this repository writes exactly these fields.

In [ ]:
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('Upload the water-level CSV exported from AHON FloodWatch.')

csv_name = next(iter(uploaded))
raw = pd.read_csv(io.BytesIO(uploaded[csv_name]))
required_columns = {'timestamp', 'water_level_cm'}
missing_columns = required_columns - set(raw.columns)
if missing_columns:
    raise ValueError(f'Missing required CSV columns: {sorted(missing_columns)}')

raw['timestamp'] = pd.to_datetime(raw['timestamp'], errors='coerce', utc=True)
raw['water_level_cm'] = pd.to_numeric(raw['water_level_cm'], errors='coerce')
print(f'Uploaded: {csv_name}')
print(f'Rows: {len(raw):,}')
print(f'Timestamp parse failures: {raw["timestamp"].isna().sum():,}')
print(f'Missing/non-numeric levels: {raw["water_level_cm"].isna().sum():,}')
print(f'Duplicate timestamps: {raw["timestamp"].duplicated().sum():,}')
print(f'Raw recorded range: {raw["water_level_cm"].min()} to {raw["water_level_cm"].max()}')

In [ ]:
valid_timestamp = raw['timestamp'].notna()
valid_level = raw['water_level_cm'].between(0, MAX_SENSOR_CM, inclusive='both')
valid = raw.loc[valid_timestamp & valid_level, ['timestamp', 'water_level_cm']].copy()
valid = valid.drop_duplicates(subset='timestamp', keep='last').sort_values('timestamp')
excluded = len(raw) - len(valid)
print(f'Rows retained within 0-{MAX_SENSOR_CM:g} cm: {len(valid):,}')
print(f'Rows excluded (bad timestamp, missing level, duplicate timestamp, or outside physical range): {excluded:,}')
if valid.empty:
    raise ValueError('No valid readings remain. Check units and sensor calibration before forecasting.')
print(f'Valid time span: {valid["timestamp"].min()} to {valid["timestamp"].max()}')

level = (valid.set_index('timestamp')['water_level_cm']
         .resample(f'{RESAMPLE_SECONDS}s').median().dropna().sort_index())
print(f'Resampled observations: {len(level):,}')
print(f'Resampled time span: {level.index.min()} to {level.index.max()}')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(level.index, level.values, color='#1769aa', linewidth=1.5, label='Measured water level')
ax.axhline(WARNING_CM, color='#ed8b00', linestyle='--', label=f'Warning: {WARNING_CM:g} cm')
ax.axhline(DANGER_CM, color='#c62828', linestyle='--', label=f'Critical/Danger: {DANGER_CM:g} cm')
ax.set(title='AHON FloodWatch water-level history', xlabel='Time (UTC)', ylabel='Water level (cm)')
ax.set_ylim(bottom=0, top=max(MAX_SENSOR_CM, float(level.max()) * 1.05))
ax.grid(alpha=0.25)
ax.legend()
fig.autofmt_xdate()
plt.show()

## 2. Prepare a leakage-aware regression dataset

Features use only current and previous readings. The target is the water level at the configured future horizon. Long gaps are split into separate segments so lag windows do not silently bridge missing periods. Train/test separation is chronological, not shuffled.

In [ ]:
interval_minutes = RESAMPLE_SECONDS / 60
horizon_steps = max(1, int(round(FORECAST_MINUTES / interval_minutes)))
gap_limit = pd.Timedelta(seconds=RESAMPLE_SECONDS * GAP_MULTIPLIER)
segment_id = level.index.to_series().diff().gt(gap_limit).cumsum()
feature_frames = []
feature_columns = ['level_now', 'lag_1', 'lag_2', 'lag_4', 'lag_10', 'lag_20', 'mean_5', 'mean_10']

for _, segment in level.groupby(segment_id):
    part = pd.DataFrame(index=segment.index)
    part['level_now'] = segment
    for lag in [1, 2, 4, 10, 20]:
        part[f'lag_{lag}'] = segment.shift(lag)
    part['mean_5'] = segment.rolling(5, min_periods=5).mean()
    part['mean_10'] = segment.rolling(10, min_periods=10).mean()
    part['target_cm'] = segment.shift(-horizon_steps)
    part['target_time'] = pd.Series(segment.index, index=segment.index).shift(-horizon_steps)
    feature_frames.append(part.dropna())

if not feature_frames:
    raise ValueError('Not enough continuous history to build lagged forecast rows.')
supervised = pd.concat(feature_frames).sort_index()
print(f'Supervised rows: {len(supervised):,}')
print(f'Lag history: up to {20 * interval_minutes:g} minutes')
print(f'Target horizon: {horizon_steps * interval_minutes:g} minutes')
if len(supervised) < MIN_SUPERVISED_ROWS:
    raise ValueError(f'Only {len(supervised)} valid supervised rows; collect more continuous readings (minimum {MIN_SUPERVISED_ROWS}) before reporting model metrics.')

In [ ]:
split_position = int(len(supervised) * (1 - TEST_FRACTION))
split_time = supervised['target_time'].iloc[split_position]
train = supervised[supervised['target_time'] < split_time]
test = supervised[supervised['target_time'] >= split_time]
X_train, y_train = train[feature_columns], train['target_cm']
X_test, y_test = test[feature_columns], test['target_cm']
if len(train) < 20 or len(test) < 10:
    raise ValueError(f'Insufficient chronological split: train={len(train)}, test={len(test)}. Collect more readings.')

model = RandomForestRegressor(
    n_estimators=200, max_depth=8, min_samples_leaf=3,
    random_state=42, n_jobs=-1
)
model.fit(X_train, y_train)
predicted = model.predict(X_test)
persistence_predicted = X_test['level_now'].to_numpy()

def metric_row(actual, estimate):
    return {
        'MAE (cm)': mean_absolute_error(actual, estimate),
        'RMSE (cm)': np.sqrt(mean_squared_error(actual, estimate)),
        'R-squared': r2_score(actual, estimate),
    }

metrics = pd.DataFrame.from_dict(
    {
        'Random Forest regression': metric_row(y_test, predicted),
        'Persistence baseline (latest level)': metric_row(y_test, persistence_predicted),
    },
    orient='index',
)
print(f'Train rows: {len(train):,}; test rows: {len(test):,}')
print(f'Test target period: {test["target_time"].min()} to {test["target_time"].max()}')
display(metrics.round(4))
print('Lower MAE/RMSE is better. Negative R-squared means worse than predicting the test-set mean.')
print('These chronological holdout metrics are exploratory and do not establish emergency forecast accuracy.')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test['target_time'], y_test.values, color='#1769aa', label='Actual')
ax.plot(test['target_time'], predicted, color='#c62828', linestyle='--', label='Random Forest regression')
ax.plot(test['target_time'], persistence_predicted, color='#2e7d32', linestyle='-.', label='Persistence baseline')
ax.axhline(WARNING_CM, color='#ed8b00', linestyle=':', label='Warning boundary')
ax.axhline(DANGER_CM, color='#8e0000', linestyle=':', label='Danger boundary')
ax.set(title=f'{FORECAST_MINUTES}-minute-ahead forecast on chronological test data', xlabel='Forecast target time (UTC)', ylabel='Water level (cm)')
ax.grid(alpha=0.25)
ax.legend()
fig.autofmt_xdate()
plt.show()

In [ ]:
latest_segment_id = segment_id.iloc[-1]
latest_segment = level[segment_id == latest_segment_id]
latest_features = pd.DataFrame(index=[latest_segment.index[-1]])
latest_features['level_now'] = [latest_segment.iloc[-1]]
for lag in [1, 2, 4, 10, 20]:
    if len(latest_segment) <= lag:
        raise ValueError('Latest continuous segment is too short for forecast lags.')
    latest_features[f'lag_{lag}'] = [latest_segment.iloc[-lag - 1]]
latest_features['mean_5'] = [latest_segment.iloc[-5:].mean()]
latest_features['mean_10'] = [latest_segment.iloc[-10:].mean()]
next_level = float(model.predict(latest_features[feature_columns])[0])
forecast_time = latest_segment.index[-1] + pd.Timedelta(minutes=FORECAST_MINUTES)
print(f'Latest measured level: {latest_segment.iloc[-1]:.2f} cm at {latest_segment.index[-1]}')
print(f'Exploratory forecast: {next_level:.2f} cm at {forecast_time} (UTC)')
fig, ax = plt.subplots(figsize=(12, 4))
recent = latest_segment.tail(int(30 * 60 / RESAMPLE_SECONDS))
ax.plot(recent.index, recent.values, color='#1769aa', label='Recent measured readings')
ax.scatter([forecast_time], [next_level], color='#c62828', marker='x', s=90, label='Forecast point')
ax.axhline(WARNING_CM, color='#ed8b00', linestyle='--', label='Warning: 2 cm')
ax.axhline(DANGER_CM, color='#8e0000', linestyle='--', label='Danger: 3 cm')
ax.set(title='Short-horizon demonstration forecast', xlabel='Time (UTC)', ylabel='Water level (cm)')
ax.grid(alpha=0.25)
ax.legend()
fig.autofmt_xdate()
plt.show()

## Defense notes and limitations

- Present the recorded-data count, date range, excluded out-of-range count, and test-period metrics shown by the notebook.
- Explain that the target is measured water level in centimeters at a future time; this is regression, not flood-status classification.
- Do not describe R-squared as percentage accuracy. MAE/RMSE are errors in centimeters.
- The dataset is short and may not represent different rain, tide, or flood conditions. Keep operational warnings and evacuation decisions based on the validated sensor thresholds and responders, not this exploratory model alone.